In [ ]:
import gzip
import json
import shutil
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import pysam
from IPython.display import HTML, clear_output, display

FRAGMENTS = pd.DataFrame([
    {"chromosome": "Pf3D7_02_v3", "start_position": 273881,  "end_position": 274429,  "gene_symbol": "msp2",        "gene_id": "PF3D7_0206800", "marker_type": "genetic diversity marker"},
    {"chromosome": "Pf3D7_04_v3", "start_position": 532148,  "end_position": 532365,  "gene_symbol": "polyA",       "gene_id": "PF3D7_0411900", "marker_type": "genetic diversity marker"},
    {"chromosome": "Pf3D7_04_v3", "start_position": 748109,  "end_position": 748599,  "gene_symbol": "dhfr",        "gene_id": "PF3D7_0417200", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_05_v3", "start_position": 958117,  "end_position": 958480,  "gene_symbol": "mdr1",        "gene_id": "PF3D7_0523000", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_05_v3", "start_position": 961287,  "end_position": 962028,  "gene_symbol": "mdr1",        "gene_id": "PF3D7_0523000", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_06_v3", "start_position": 899663,  "end_position": 900082,  "gene_symbol": "ta1",         "gene_id": "PF3D7_0622100", "marker_type": "genetic diversity marker"},
    {"chromosome": "Pf3D7_07_v3", "start_position": 403516,  "end_position": 403693,  "gene_symbol": "crt",         "gene_id": "PF3D7_0709000", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_07_v3", "start_position": 405185,  "end_position": 405634,  "gene_symbol": "crt",         "gene_id": "PF3D7_0709000", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_08_v3", "start_position": 549597,  "end_position": 550238,  "gene_symbol": "dhps",        "gene_id": "PF3D7_0810800", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_09_v3", "start_position": 1201915, "end_position": 1202211, "gene_symbol": "msp1",        "gene_id": "PF3D7_0930300", "marker_type": "genetic diversity marker"},
    {"chromosome": "Pf3D7_12_v3", "start_position": 2092339, "end_position": 2093226, "gene_symbol": "coronin",     "gene_id": "PF3D7_1251200", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_13_v3", "start_position": 748259,  "end_position": 748560,  "gene_symbol": "ferredoxin",  "gene_id": "PF3D7_1318100", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_13_v3", "start_position": 1724853, "end_position": 1725702, "gene_symbol": "k13",         "gene_id": "PF3D7_1343700", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_13_v3", "start_position": 2504486, "end_position": 2504727, "gene_symbol": "exonuclease", "gene_id": "PF3D7_1362500", "marker_type": "drug resistance marker"},
    {"chromosome": "Pf3D7_14_v3", "start_position": 1956099, "end_position": 1956640, "gene_symbol": "mdr2",        "gene_id": "PF3D7_1447900", "marker_type": "drug resistance marker"},
])

FRAGMENTS["fragment_id"] = FRAGMENTS.apply(
    lambda r: f"fragment_{r['gene_symbol']}_{r['chromosome']}_{r['start_position']}_{r['end_position']}",
    axis=1,
)
FRAGMENTS = FRAGMENTS[["fragment_id", "chromosome", "start_position", "end_position", "gene_symbol", "gene_id", "marker_type"]]

PASS_THRESHOLD = 100
AUTO_REFRESH_SECONDS = 10
RERUN_DELAY_SECONDS = 10

PROJECT_ROOT = Path.cwd()
REFERENCE_FASTA = (PROJECT_ROOT / "Reference" / "Pf3D7_v2.fasta").resolve()

OUTPUT_PARENT = Path("/tmp")
OUTPUT_PREFIX = "output_"

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def timestamp_str() -> str:
    return datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

def tool_exists(name: str) -> bool:
    return shutil.which(name) is not None

def validate_runtime() -> List[str]:
    missing = [x for x in ["minimap2", "samtools"] if not tool_exists(x)]
    problems = []
    if missing:
        problems.append("Missing required tools in PATH: " + ", ".join(missing))
    if not REFERENCE_FASTA.exists():
        problems.append(f"Reference FASTA not found: {REFERENCE_FASTA}")
    return problems

def list_read_files(barcode_dir: Path) -> List[Path]:
    files: List[Path] = []
    for pattern in ["*.fastq", "*.fastq.gz", "*.fq", "*.fq.gz"]:
        files.extend(sorted(barcode_dir.glob(pattern)))
    return files

def discover_barcodes(fastq_pass_dir: Path) -> List[Path]:
    return sorted([p for p in fastq_pass_dir.glob("barcode*") if p.is_dir()])

def concat_fastq(inputs: List[Path], output_gz: Path) -> None:
    output_gz.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(output_gz, "wb") as out_handle:
        for fp in inputs:
            if fp.suffix == ".gz":
                with gzip.open(fp, "rb") as in_handle:
                    shutil.copyfileobj(in_handle, out_handle)
            else:
                with open(fp, "rb") as in_handle:
                    shutil.copyfileobj(in_handle, out_handle)

def style_read_counts(df: pd.DataFrame):
    def color_value(v):
        try:
            return "color: #c62828; font-weight: 700;" if float(v) < PASS_THRESHOLD else "color: #1d4ed8; font-weight: 700;"
        except Exception:
            return ""
    subset = [c for c in df.columns if c != "barcode"]
    return (
        df.style.hide(axis="index")
        .set_properties(subset=["barcode"], **{"font-weight": "700"})
        .map(color_value, subset=subset)
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#eff6ff"), ("font-weight", "700")]},
            {"selector": "td, th", "props": [("border", "1px solid #d7e0ec"), ("padding", "6px 8px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        ])
    )

def style_summary(df: pd.DataFrame):
    def color_pct(v):
        try:
            return "color: #c62828; font-weight: 700;" if float(v) < 100 else "color: #0f766e; font-weight: 700;"
        except Exception:
            return ""
    return (
        df.style.hide(axis="index")
        .format({"pass_percentage": "{:.2f}"})
        .map(color_pct, subset=["pass_percentage"])
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#eff6ff"), ("font-weight", "700")]},
            {"selector": "td, th", "props": [("border", "1px solid #d7e0ec"), ("padding", "6px 8px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        ])
    )

def make_fragment_figure(read_counts_df: pd.DataFrame, fragment_id: str) -> go.Figure:
    local_df = read_counts_df[["barcode", fragment_id]].copy().sort_values("barcode")
    colors = ["#1d4ed8" if v >= PASS_THRESHOLD else "#c62828" for v in local_df[fragment_id]]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=local_df["barcode"],
        y=local_df[fragment_id],
        marker_color=colors,
        hovertemplate="Barcode=%{x}<br>Reads=%{y}<extra></extra>",
        name="Reads",
    ))
    fig.add_hline(y=PASS_THRESHOLD, line_color="#c62828", line_width=2, line_dash="dash")
    fig.update_layout(
        title=fragment_id,
        template="plotly_white",
        height=360,
        margin=dict(l=30, r=20, t=60, b=60),
        xaxis_title="Barcode",
        yaxis_title="Read count",
        showlegend=False,
    )
    fig.update_xaxes(tickangle=-45)
    return fig

class RunPaths:
    def __init__(self, root: Path):
        self.root = root
        self.run_dir = self.root / "run"
        self.res_dir = self.root / "res"
        self.log_file = self.res_dir / "dashboard.log"
        self.status_file = self.res_dir / "status.json"
        self.read_counts_csv = self.res_dir / "read_counts.csv"
        self.summary_csv = self.res_dir / "summary.csv"
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.res_dir.mkdir(parents=True, exist_ok=True)

class FragmentDashboard:
    def __init__(self):
        self.worker_thread: Optional[threading.Thread] = None
        self.stop_event = threading.Event()
        self.is_running = False
        self.current_paths: Optional[RunPaths] = None
        self.active_fastq_pass: Optional[Path] = None
        self.run_cycle_index = 0

        self.fastq_pass_input = widgets.Text(
            value="",
            placeholder="/path/to/fastq_pass",
            description="fastq_pass:",
            layout=widgets.Layout(width="99%"),
        )
        self.file_picker = widgets.FileUpload(
            accept="",
            multiple=False
        )
        self.threads_input = widgets.BoundedIntText(
            value=10,
            min=1,
            max=64,
            description="Threads:",
            layout=widgets.Layout(width="220px"),
        )
        self.run_button = widgets.Button(description="Run monitoring", button_style="primary", layout=widgets.Layout(width="170px", height="42px"))
        self.stop_button = widgets.Button(description="Stop monitoring", button_style="warning", layout=widgets.Layout(width="170px", height="42px"))
        self.refresh_button = widgets.Button(description="Refresh now", layout=widgets.Layout(width="150px", height="42px"))

        self.spinner = widgets.HTML(value=self._spinner_html(False))
        self.reference_html = widgets.HTML()
        self.status_html = widgets.HTML()
        self.metrics_html = widgets.HTML()
        self.output_html = widgets.HTML()
        self.log_area = widgets.Textarea(layout=widgets.Layout(width="100%", height="220px"))
        self.tables_output = widgets.Output()
        self.figures_output = widgets.Output()
        self.main_tabs = widgets.Tab(children=[self.tables_output, self.figures_output])
        self.main_tabs.set_title(0, "Tables")
        self.main_tabs.set_title(1, "Figures")

        self.run_button.on_click(self._on_run_clicked)
        self.stop_button.on_click(self._on_stop_clicked)
        self.refresh_button.on_click(self._on_refresh_clicked)
        self.file_picker.observe(self._on_file_upload_change, names="value")

        self._auto_refresh_thread = threading.Thread(target=self._auto_refresh_loop, daemon=True)
        self._auto_refresh_thread.start()

        self._render_layout()
        self.refresh_dashboard()

    def _make_run_paths(self) -> RunPaths:
        return RunPaths((OUTPUT_PARENT / f"{OUTPUT_PREFIX}{timestamp_str()}").resolve())

    def _append_log(self, message: str) -> None:
        if self.current_paths is None:
            return
        with open(self.current_paths.log_file, "a", encoding="utf-8") as handle:
            handle.write(f"[{now_str()}] {message}\n")

    def _save_status(self, payload: Dict) -> None:
        if self.current_paths is None:
            return
        payload = dict(payload)
        payload["updated_at"] = now_str()
        with open(self.current_paths.status_file, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2)

    def _load_status(self) -> Dict:
        if self.current_paths is None:
            return {"state": "idle", "message": "Waiting for first run.", "updated_at": now_str()}
        if self.current_paths.status_file.exists():
            return json.loads(self.current_paths.status_file.read_text(encoding="utf-8"))
        return {"state": "idle", "message": "Waiting for first run.", "updated_at": now_str()}

    def _read_counts_df(self) -> pd.DataFrame:
        if self.current_paths is None:
            return pd.DataFrame(columns=["barcode"])
        if self.current_paths.read_counts_csv.exists():
            try:
                return pd.read_csv(self.current_paths.read_counts_csv)
            except Exception:
                return pd.DataFrame(columns=["barcode"])
        return pd.DataFrame(columns=["barcode"])

    def _summary_df(self) -> pd.DataFrame:
        if self.current_paths is None:
            return pd.DataFrame(columns=["fragment_id", "total_samples", "pass_samples", "pass_percentage"])
        if self.current_paths.summary_csv.exists():
            try:
                return pd.read_csv(self.current_paths.summary_csv)
            except Exception:
                return pd.DataFrame(columns=["fragment_id", "total_samples", "pass_samples", "pass_percentage"])
        return pd.DataFrame(columns=["fragment_id", "total_samples", "pass_samples", "pass_percentage"])

    def _compute_summary(self, read_counts_df: pd.DataFrame) -> pd.DataFrame:
        fragment_columns = [c for c in read_counts_df.columns if c != "barcode"]
        n_total = len(read_counts_df)
        rows = []
        for fragment in fragment_columns:
            n_pass = int((read_counts_df[fragment] >= PASS_THRESHOLD).sum()) if n_total else 0
            pct = (n_pass / n_total * 100.0) if n_total else 0.0
            rows.append({
                "fragment_id": fragment,
                "total_samples": n_total,
                "pass_samples": n_pass,
                "pass_percentage": pct,
            })
        return pd.DataFrame(rows)

    def _write_partial_results(self, records: List[Dict]) -> None:
        if self.current_paths is None:
            return
        if records:
            read_counts_df = pd.DataFrame(records).sort_values("barcode").reset_index(drop=True)
            summary_df = self._compute_summary(read_counts_df)
        else:
            read_counts_df = pd.DataFrame(columns=["barcode"])
            summary_df = pd.DataFrame(columns=["fragment_id", "total_samples", "pass_samples", "pass_percentage"])
        read_counts_df.to_csv(self.current_paths.read_counts_csv, index=False)
        summary_df.to_csv(self.current_paths.summary_csv, index=False)

    def _cleanup_old_outputs(self) -> None:
        current_root = self.current_paths.root if self.current_paths is not None else None
        for path in OUTPUT_PARENT.glob(f"{OUTPUT_PREFIX}*"):
            if not path.is_dir():
                continue
            if current_root is not None and path.resolve() == current_root.resolve():
                continue
            try:
                shutil.rmtree(path)
            except Exception:
                pass

    def _build_barcode_bam(self, barcode_dir: Path, threads: int = 10) -> Path:
        if self.current_paths is None:
            raise RuntimeError("No active output directory.")
        barcode = barcode_dir.name
        concat_gz = self.current_paths.run_dir / f"{barcode}.fastq.gz"
        bam_path = self.current_paths.run_dir / f"{barcode}.bam"
        bai_path = self.current_paths.run_dir / f"{barcode}.bam.bai"

        inputs = list_read_files(barcode_dir)
        if not inputs:
            raise FileNotFoundError(f"No FASTQ files found in {barcode_dir}")

        concat_fastq(inputs, concat_gz)

        self._append_log(
            "CMD: minimap2 -t {} -ax map-ont '{}' '{}' | samtools sort -o '{}'".format(
                threads, REFERENCE_FASTA, concat_gz, bam_path
            )
        )

        p1 = subprocess.Popen(
            [
                "minimap2",
                "-t", str(threads),
                "-ax", "map-ont",
                str(REFERENCE_FASTA),
                str(concat_gz),
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False,
        )

        p2 = subprocess.Popen(
            [
                "samtools",
                "sort",
                "-o", str(bam_path),
            ],
            stdin=p1.stdout,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False,
        )

        if p1.stdout is not None:
            p1.stdout.close()

        _, samtools_stderr = p2.communicate()
        minimap2_stderr = p1.stderr.read() if p1.stderr is not None else b""
        p1_returncode = p1.wait()
        p2_returncode = p2.returncode

        if minimap2_stderr:
            self._append_log("STDERR minimap2: " + minimap2_stderr.decode(errors="replace").strip())
        if samtools_stderr:
            self._append_log("STDERR samtools: " + samtools_stderr.decode(errors="replace").strip())

        if p1_returncode != 0:
            raise RuntimeError(f"minimap2 failed with exit code {p1_returncode}")
        if p2_returncode != 0:
            raise RuntimeError(f"samtools sort failed with exit code {p2_returncode}")

        completed = subprocess.run(
            ["samtools", "index", str(bam_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        self._append_log("CMD: samtools index " + str(bam_path))
        if completed.stdout.strip():
            self._append_log("STDOUT: " + completed.stdout.strip())
        if completed.stderr.strip():
            self._append_log("STDERR: " + completed.stderr.strip())
        if completed.returncode != 0:
            raise RuntimeError(f"samtools index failed with exit code {completed.returncode}")

        if not bam_path.exists() or not bai_path.exists():
            raise RuntimeError(f"BAM or BAI file was not generated for {barcode}")

        return bam_path

    def _count_reads_per_fragment(self, bam_path: Path) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        with pysam.AlignmentFile(str(bam_path), "rb") as bam:
            for _, row in FRAGMENTS.iterrows():
                counts[row["fragment_id"]] = int(sum(
                    1 for _ in bam.fetch(row["chromosome"], int(row["start_position"]), int(row["end_position"]))
                ))
        return counts

    def _on_file_upload_change(self, change):
        value = change.get("new", {})
        if not value:
            return
        try:
            first_key = next(iter(value))
            uploaded = value[first_key]
            name = uploaded.get("metadata", {}).get("name", "") or first_key
            self.fastq_pass_input.value = name
        except Exception:
            pass

    def _auto_refresh_loop(self):
        while True:
            time.sleep(AUTO_REFRESH_SECONDS)
            try:
                self.refresh_dashboard()
            except Exception:
                pass

    def _spinner_html(self, active: bool) -> str:
        if not active:
            return "<div class='dashboard-card'><span class='run-pill'>Idle</span><span class='dashboard-subtle'>No active backend job.</span></div>"
        return """
        <div class='dashboard-card'>
            <span class='run-pill'>Running</span>
            <span class='dashboard-subtle'>The backend is updating BAM files and fragment counts automatically.</span>
            <div style='margin-top:10px;'>
                <div style='width:26px;height:26px;border:4px solid #dbeafe;border-top:4px solid #1d4ed8;border-radius:50%;animation: spin 1s linear infinite;'></div>
            </div>
            <style>@keyframes spin { from { transform: rotate(0deg);} to { transform: rotate(360deg);} }</style>
        </div>
        """

    def _render_layout(self):
        display(HTML("""
        <style>
            .dashboard-banner { background: linear-gradient(135deg, #0f172a 0%, #1d4ed8 100%); color: white; border-radius: 18px; padding: 22px 26px; margin-bottom: 16px; box-shadow: 0 12px 30px rgba(15, 23, 42, 0.20); }
            .dashboard-card { background: white; border: 1px solid #d7e0ec; border-radius: 16px; padding: 14px 16px; margin-bottom: 12px; box-shadow: 0 6px 18px rgba(15, 23, 42, 0.06); }
            .dashboard-section-title { font-size: 16px; font-weight: 800; color: #0f172a; margin-bottom: 6px; }
            .dashboard-subtle { color: #5b6b81; font-size: 13px; }
            .metric-box { background: #f8fbff; border: 1px solid #dbeafe; border-radius: 14px; padding: 14px; min-width: 170px; margin: 6px; }
            .metric-number { font-size: 28px; font-weight: 800; color: #0f172a; }
            .metric-label { color: #5b6b81; font-size: 12px; margin-top: 4px; }
            .run-pill { display: inline-block; background: #dbeafe; color: #1d4ed8; font-weight: 800; border-radius: 999px; padding: 5px 12px; margin-right: 8px; }
        </style>
        """))
        display(HTML("""
        <div class='dashboard-banner'>
            <div style='display:flex;align-items:center;justify-content:space-between;gap:20px;'>
                <div>
                    <div style='font-size:28px;font-weight:800;'>ONT Fragment Monitoring Dashboard - Malaria team</div>
                    <div style='margin-top:6px;font-size:14px;opacity:0.95;'>Real-time barcode and fragment monitoring during sequencing with BAM-level processing only.</div>
                    <div style='margin-top:6px;font-size:14px;opacity:0.95;'>IGH (Institute of Genomics and Global Health)</div>
                </div>
                <div>
                    <img src='img/logo.png' alt='IGH Logo' style='height:150px;width:auto;object-fit:contain;' />
                </div>
            </div>
        </div>
        """))
        controls = widgets.VBox([
            widgets.HTML("<div class='dashboard-card'><div class='dashboard-section-title'>Run configuration</div><div class='dashboard-subtle'>Select the ONT fastq_pass directory. The reference is fixed to Reference/Pf3D7_v2.fasta.</div></div>"),
            self.fastq_pass_input,
            self.file_picker,
            widgets.HBox([self.threads_input]),
            widgets.HBox([self.run_button, self.stop_button, self.refresh_button]),
            self.output_html,
            widgets.HTML("<div class='dashboard-card'><div class='dashboard-section-title'>Backend log</div></div>"),
            self.log_area,
        ], layout=widgets.Layout(width="98%"))

        display(widgets.HBox([
            widgets.VBox([controls], layout=widgets.Layout(width="52%")),
            widgets.VBox([self.spinner, self.reference_html, self.status_html, self.metrics_html], layout=widgets.Layout(width="48%")),
        ], layout=widgets.Layout(width="100%")))

        display(HTML("<div class='dashboard-card'><div class='dashboard-section-title'>Results</div><div class='dashboard-subtle'>Tables and dynamic Plotly figures refresh automatically every 10 seconds and update while the run is progressing.</div></div>"))
        display(self.main_tabs)

    def _on_run_clicked(self, _):
        if self.is_running:
            self.refresh_dashboard()
            return

        fastq_pass = Path(self.fastq_pass_input.value).expanduser().resolve()
        if not fastq_pass.exists() or fastq_pass.name != "fastq_pass":
            self.status_html.value = "<div class='dashboard-card'><span style='color:#c62828;font-weight:700;'>Invalid input:</span> the selected path must be an existing fastq_pass directory.</div>"
            return

        problems = validate_runtime()
        if problems:
            self.status_html.value = f"<div class='dashboard-card'><span style='color:#c62828;font-weight:700;'>Runtime error:</span><br>{'<br>'.join(problems)}</div>"
            return

        self.active_fastq_pass = fastq_pass
        self.run_cycle_index = 0
        self.stop_event.clear()
        self.is_running = True
        self.spinner.value = self._spinner_html(True)
        self.worker_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.worker_thread.start()
        self.refresh_dashboard()

    def _on_stop_clicked(self, _):
        self.stop_event.set()
        self.is_running = False
        self._save_status({"state": "stopped", "message": "Monitoring stopped by user."})
        self._append_log("Monitoring stopped by user.")
        self.spinner.value = self._spinner_html(False)
        self.refresh_dashboard()

    def _on_refresh_clicked(self, _):
        self.refresh_dashboard()

    def _monitor_loop(self):
        try:
            while not self.stop_event.is_set():
                if self.active_fastq_pass is None:
                    break

                self.current_paths = self._make_run_paths()
                self.run_cycle_index += 1
                self._write_partial_results([])
                self._save_status({"state": "running", "message": f"Preparing run {self.run_cycle_index} in {self.current_paths.root}"})
                self._append_log(f"Starting monitoring for {self.active_fastq_pass}")
                self._run_one_cycle(self.active_fastq_pass)

                if self.stop_event.is_set():
                    break

                self._save_status({"state": "completed", "message": f"Run {self.run_cycle_index} completed successfully."})
                self._append_log(f"Run {self.run_cycle_index} completed successfully.")
                self.refresh_dashboard()
                self._cleanup_old_outputs()

                waited = 0
                while waited < RERUN_DELAY_SECONDS and not self.stop_event.is_set():
                    time.sleep(1)
                    waited += 1

            if self.stop_event.is_set():
                self._save_status({"state": "stopped", "message": "Monitoring stopped by user."})
        except Exception as exc:
            self._append_log(f"FATAL: {exc}")
            self._save_status({"state": "error", "message": str(exc)})
        finally:
            self.is_running = False
            self.spinner.value = self._spinner_html(False)
            self.refresh_dashboard()

    def _run_one_cycle(self, fastq_pass: Path):
        barcode_dirs = discover_barcodes(fastq_pass)
        if not barcode_dirs:
            raise RuntimeError(f"No barcode directories were found in {fastq_pass}")

        records: List[Dict] = []
        self._save_status({"state": "running", "message": f"Processing {len(barcode_dirs)} barcode directories."})
        self._append_log(f"Cycle started for {len(barcode_dirs)} barcode directories.")
        self._write_partial_results(records)
        self.refresh_dashboard()

        for idx, barcode_dir in enumerate(barcode_dirs, start=1):
            if self.stop_event.is_set():
                self._append_log("Stop request received during cycle.")
                self._write_partial_results(records)
                self.refresh_dashboard()
                return

            barcode = barcode_dir.name
            self._save_status({"state": "running", "message": f"Processing {barcode} ({idx}/{len(barcode_dirs)})."})
            self._append_log(f"Processing {barcode}")

            bam_path = self._build_barcode_bam(barcode_dir, threads=int(self.threads_input.value))
            counts = self._count_reads_per_fragment(bam_path)

            row = {"barcode": barcode}
            row.update(counts)
            records.append(row)
            self._write_partial_results(records)
            self.refresh_dashboard()

        self._write_partial_results(records)
        summary_df = self._summary_df()
        msg = f"Cycle completed successfully for {len(barcode_dirs)} barcodes."
        if len(summary_df) > 0 and (summary_df["pass_percentage"] == 100).all():
            msg = "All monitored barcode-fragment combinations have passed the threshold."
        self._save_status({"state": "completed", "message": msg})
        self._append_log("Cycle completed successfully.")
        self.refresh_dashboard()

    def refresh_dashboard(self):
        status = self._load_status()
        read_counts_df = self._read_counts_df()
        summary_df = self._summary_df()

        total_barcodes = len(read_counts_df)
        total_fragments = max(0, len(read_counts_df.columns) - 1)
        total_pass_cells = 0
        total_cells = 0
        recommendation = "Continue sequencing"

        if total_barcodes and total_fragments:
            data_only = read_counts_df.drop(columns=["barcode"])
            total_pass_cells = int((data_only >= PASS_THRESHOLD).sum().sum())
            total_cells = int(data_only.size)
            if total_cells > 0 and total_pass_cells == total_cells:
                recommendation = "All monitored cells passed. You can consider stopping sequencing."

        status_color = "#0f766e" if status.get("state") in {"running", "completed"} else ("#c62828" if status.get("state") == "error" else "#5b6b81")

        if self.current_paths is not None:
            self.output_html.value = f"<div class='dashboard-card'><div class='dashboard-section-title'>Current output</div><div class='dashboard-subtle'><b>{self.current_paths.root}</b></div></div>"
        else:
            self.output_html.value = "<div class='dashboard-card'><div class='dashboard-section-title'>Current output</div><div class='dashboard-subtle'>No active output directory yet.</div></div>"

        self.reference_html.value = f"<div class='dashboard-card'><div class='dashboard-section-title'>Reference</div><div class='dashboard-subtle'><b>Fixed path:</b> {REFERENCE_FASTA}</div></div>"
        self.status_html.value = f"""
        <div class='dashboard-card'>
            <div class='dashboard-section-title'>Run status</div>
            <div><span style='font-weight:700;color:{status_color};'>{status.get('state', 'idle').upper()}</span></div>
            <div class='dashboard-subtle' style='margin-top:6px;'>{status.get('message', '')}</div>
            <div class='dashboard-subtle' style='margin-top:6px;'>Last update: {status.get('updated_at', '')}</div>
            <div style='margin-top:10px;font-weight:700;color:#0f172a;'>{recommendation}</div>
        </div>
        """
        self.metrics_html.value = f"""
        <div class='dashboard-card'>
            <div class='dashboard-section-title'>Current metrics</div>
            <div style='display:flex;flex-wrap:wrap;'>
                <div class='metric-box'><div class='metric-number'>{total_barcodes}</div><div class='metric-label'>Barcodes detected</div></div>
                <div class='metric-box'><div class='metric-number'>{total_fragments}</div><div class='metric-label'>Fragments monitored</div></div>
                <div class='metric-box'><div class='metric-number'>{total_pass_cells}</div><div class='metric-label'>Barcode-fragment passes</div></div>
                <div class='metric-box'><div class='metric-number'>{total_cells}</div><div class='metric-label'>Barcode-fragment observations</div></div>
            </div>
        </div>
        """

        if self.current_paths is not None and self.current_paths.log_file.exists():
            self.log_area.value = self.current_paths.log_file.read_text(encoding="utf-8")[-12000:]
        else:
            self.log_area.value = ""

        with self.tables_output:
            clear_output(wait=True)
            read_counts_out = widgets.Output()
            summary_out = widgets.Output()

            with read_counts_out:
                display(HTML("<div class='dashboard-card'><div class='dashboard-section-title'>read_counts</div><div class='dashboard-subtle'>Values below 100 are red. Values at or above 100 are blue.</div></div>"))
                if len(read_counts_df):
                    display(style_read_counts(read_counts_df))
                else:
                    display(HTML("<div class='dashboard-card'>No read count results are available yet.</div>"))

            with summary_out:
                display(HTML("<div class='dashboard-card'><div class='dashboard-section-title'>summary</div><div class='dashboard-subtle'>Fragment-level pass rate across all barcodes.</div></div>"))
                if len(summary_df):
                    display(style_summary(summary_df))
                else:
                    display(HTML("<div class='dashboard-card'>No summary results are available yet.</div>"))

            tabs = widgets.Tab(children=[read_counts_out, summary_out])
            tabs.set_title(0, "read_counts")
            tabs.set_title(1, "summary")
            display(tabs)

        with self.figures_output:
            clear_output(wait=True)
            display(HTML("<div class='dashboard-card'><div class='dashboard-section-title'>Fragment figures</div><div class='dashboard-subtle'>One Plotly figure per fragment, displayed two per row, with a threshold line at 100 reads.</div></div>"))
            if len(read_counts_df) and len(read_counts_df.columns) > 1:
                fragment_ids = [c for c in read_counts_df.columns if c != "barcode"]
                panels = []
                for fragment_id in fragment_ids:
                    out = widgets.Output(layout=widgets.Layout(width="49%", border="1px solid #d7e0ec", padding="8px"))
                    with out:
                        make_fragment_figure(read_counts_df, fragment_id).show()
                    panels.append(out)
                rows = []
                for i in range(0, len(panels), 2):
                    rows.append(widgets.HBox(panels[i:i+2], layout=widgets.Layout(width="100%", justify_content="space-between")))
                display(widgets.VBox(rows))
            else:
                display(HTML("<div class='dashboard-card'>No figure can be generated yet because no read counts are available.</div>"))

dashboard = FragmentDashboard()